In [ ]:
"""
YouTube Audio Download & Transcription Pipeline (Separated Workflow)
=====================================================================
Workflow:
1. Download all audio files and save metadata to database (no transcription yet)
2. Transcribe all audio files in database that have audio_loc but no text_loc

This allows:
- Download when internet is good (doesn't need GPU)
- Transcribe later when GPU is free
- Resume easily if interrupted

Usage:
    python main.py

Requirements:
    pip install yt-dlp transformers torch tqdm
"""

In [ ]:
import os
import sqlite3
import json
import time
import threading
from typing import Tuple, Optional, List
from tqdm import tqdm
import yt_dlp

In [ ]:
# ============================================================================
# FFmpeg Setup - Add to PATH before importing torch/transformers
# ============================================================================
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
FFMPEG_PATH = os.path.join(
    SCRIPT_DIR, "ffmpeg-2025-03-31-git-35c091f4b7-full_build", "bin"
)

In [ ]:
if os.path.exists(FFMPEG_PATH) and FFMPEG_PATH not in os.environ.get("PATH", ""):
    os.environ["PATH"] = FFMPEG_PATH + os.pathsep + os.environ.get("PATH", "")
    print(f"✅ Added ffmpeg to PATH: {FFMPEG_PATH}")

In [ ]:
import torch

In [ ]:
# ============================================================================
# CUDA / Device Setup
# ============================================================================
if torch.cuda.is_available():
    cuda_device = torch.cuda.current_device()
    print(f"🚀 CUDA Device: {torch.cuda.get_device_name(cuda_device)}")
    DEVICE = "cuda"
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    print("⚠️ CUDA is not available, running on CPU (slower).")
    DEVICE = "cpu"

In [ ]:
# ============================================================================
# Configuration
# ============================================================================
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.abspath(os.path.join(SCRIPT_DIR, "..", ".."))
DATA_DIR = os.path.join(PROJECT_ROOT, "data")

In [ ]:
TEMP_FOLDER = os.path.join(DATA_DIR, "tempvid")
TRANSCRIBE_FOLDER = os.path.join(DATA_DIR, "transcribe_result")
DATABASE_PATH = os.path.join(DATA_DIR, "youtube.db")

In [ ]:
WHISPER_MODEL = "openai/whisper-large-v3"  # Change for speed/accuracy trade-off
MAX_RETRIES = 3
RETRY_DELAY_BASE = 5
TRANSCRIBE_LIMIT = 9999  # Maximum videos to transcribe per run

In [ ]:
os.makedirs(TEMP_FOLDER, exist_ok=True)
os.makedirs(TRANSCRIBE_FOLDER, exist_ok=True)

In [ ]:
# ============================================================================
# Whisper Model Setup (Lazy Loading - only when transcribing)
# ============================================================================
pipe = None

In [ ]:
def load_whisper_model():
    """Load Whisper model only when needed (for transcription)."""
    global pipe
    if pipe is None:
        print(f"📦 Loading Whisper model: {WHISPER_MODEL}")
        print(f"🖥️ Device: {DEVICE}")
        from transformers import pipeline as hf_pipeline

        pipe = hf_pipeline(
            "automatic-speech-recognition",
            model=WHISPER_MODEL,
            device=DEVICE,
            torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
            return_timestamps=True,
            chunk_length_s=30,
            stride_length_s=(4, 2),
            generate_kwargs={
                "task": "transcribe",
                "language": None,  # Auto-detect language
            },
        )
        print(f"✅ Model loaded successfully!")
    return pipe

============================================================================
Database Functions
============================================================================

In [ ]:
def init_db():
    """Initialize SQLite database and create tables if they don't exist."""
    conn = sqlite3.connect(DATABASE_PATH)
    cursor = conn.cursor()
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS videos (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            video_id TEXT UNIQUE,
            title TEXT,
            youtuber TEXT,
            date_created TEXT,
            text_loc TEXT,
            audio_loc TEXT,
            link TEXT UNIQUE,
            time_appended TEXT,
            fixed INTEGER DEFAULT 0
        )
    """
    )
    conn.commit()
    return conn

In [ ]:
def connect_db():
    """Connect to SQLite database."""
    return sqlite3.connect(DATABASE_PATH)

In [ ]:
def save_audio_metadata(
    cursor,
    conn,
    video_id: str,
    title: str,
    youtuber: str,
    date_created: str,
    audio_loc: str,
    link: str,
):
    """Save audio metadata to database (without text_loc)."""
    cursor.execute(
        """
        INSERT OR IGNORE INTO videos (video_id, title, youtuber, date_created, audio_loc, link, time_appended)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """,
        (
            video_id,
            title,
            youtuber,
            date_created,
            audio_loc,
            link,
            time.strftime("%Y-%m-%d %H:%M:%S"),
        ),
    )
    conn.commit()

In [ ]:
def update_transcription(cursor, conn, video_id: str, text_loc: str):
    """Update database with transcription location."""
    cursor.execute(
        """
        UPDATE videos SET text_loc = ? WHERE video_id = ?
    """,
        (text_loc, video_id),
    )
    conn.commit()

In [ ]:
def get_videos_without_transcription():
    """Get all videos that have audio_loc but no text_loc."""
    conn = connect_db()
    cursor = conn.cursor()
    cursor.execute(
        """
        SELECT video_id, audio_loc, title, youtuber 
        FROM videos 
        WHERE audio_loc IS NOT NULL AND audio_loc != '' 
        AND (text_loc IS NULL OR text_loc = '')
    """
    )
    results = cursor.fetchall()
    conn.close()
    return results

In [ ]:
def is_video_in_db(cursor, link: str) -> bool:
    """Check if video already exists in database."""
    cursor.execute("SELECT 1 FROM videos WHERE link = ?", (link,))
    return cursor.fetchone() is not None

============================================================================
Download Functions
============================================================================

In [ ]:
def retry_on_failure(func):
    """Decorator that retries a function on failure with exponential backoff."""

    def wrapper(*args, **kwargs):
        last_exception = None
        for attempt in range(MAX_RETRIES):
            try:
                return func(*args, **kwargs)
            except Exception as e:
                last_exception = e
                if attempt < MAX_RETRIES - 1:
                    delay = RETRY_DELAY_BASE * (2**attempt)
                    print(f"⚠️ Attempt {attempt + 1}/{MAX_RETRIES} failed: {e}")
                    print(f"🔄 Retrying in {delay} seconds...")
                    time.sleep(delay)
                else:
                    print(f"❌ All {MAX_RETRIES} attempts failed: {e}")
        raise last_exception

    return wrapper

In [ ]:
def get_playlist_videos(playlist_url: str) -> List[str]:
    """Extract all video URLs from a YouTube playlist."""

    @retry_on_failure
    def _extract():
        with yt_dlp.YoutubeDL(
            {"quiet": True, "extract_flat": True, "force_generic_extractor": True}
        ) as ydl:
            info = ydl.extract_info(playlist_url, download=False)
            return [entry["url"] for entry in info["entries"] if "url" in entry]

    try:
        return _extract()
    except Exception as e:
        print(f"❌ Playlist extraction failed: {e}")
        return []

In [ ]:
def download_audio(
    link: str, temp_folder: str = TEMP_FOLDER
) -> Tuple[Optional[str], Optional[str], dict]:
    """Download audio from a YouTube video with progress bar and retry support."""
    pbar = None

    def progress_hook(d):
        nonlocal pbar
        if d["status"] == "downloading":
            if pbar is None:
                total = d.get("total_bytes") or d.get("total_bytes_estimate") or 0
                pbar = tqdm(
                    total=total,
                    unit="B",
                    unit_scale=True,
                    desc="⬇️ Downloading",
                    bar_format="{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{rate_fmt}]",
                )

            downloaded = d.get("downloaded_bytes", 0)
            if pbar.total == 0 and d.get("total_bytes"):
                pbar.total = d["total_bytes"]
            pbar.n = downloaded
            pbar.refresh()

        elif d["status"] == "finished":
            if pbar:
                pbar.close()
            print("🔄 Converting to MP3...")

    @retry_on_failure
    def _download():
        os.makedirs(temp_folder, exist_ok=True)

        with yt_dlp.YoutubeDL(
            {
                "format": "bestaudio/best",
                "outtmpl": os.path.join(temp_folder, "%(id)s.%(ext)s"),
                "postprocessors": [
                    {
                        "key": "FFmpegExtractAudio",
                        "preferredcodec": "mp3",
                        "preferredquality": "192",
                    }
                ],
                "quiet": True,
                "no_warnings": True,
                "ffmpeg_location": FFMPEG_PATH,
                "continuedl": True,
                "fragment_retries": 10,
                "retries": 10,
                "socket_timeout": 30,
                "progress_hooks": [progress_hook],
            }
        ) as ydl:
            info = ydl.extract_info(link, download=True)
            video_id = info["id"]
            return (
                video_id,
                os.path.join(temp_folder, f"{video_id}.mp3"),
                {
                    "video_name": info.get("title", "Unknown"),
                    "youtuber": info.get("uploader", "Unknown"),
                    "time_created": info.get("upload_date", ""),
                },
            )

    try:
        return _download()
    except Exception as e:
        if pbar:
            pbar.close()
        print(f"❌ Download failed: {e}")
        return None, None, {}

============================================================================
Transcription Functions
============================================================================

In [ ]:
def transcribe(video_id: str, audio_path: str) -> Optional[str]:
    """Transcribe audio file using Whisper model with error handling for timestamp issues."""
    try:
        if not os.path.exists(audio_path):
            print(f"❌ Audio file not found: {audio_path}")
            return None

        # Load model only when transcribing
        model = load_whisper_model()

        print(f"🎙️ Transcribing audio: {os.path.basename(audio_path)}")
        start_time = time.time()

        try:
            # Try with timestamps first
            print("   Running Whisper model with timestamps...")
            result = model(audio_path, return_timestamps=True)
            print(f"   Model inference complete ({time.time() - start_time:.1f}s)")
        except Exception as e:
            # If timestamp prediction fails, try without timestamps
            if "ending timestamp" in str(e).lower() or "timestamp" in str(e).lower():
                print(f"⚠️ Timestamp prediction failed, retrying without timestamps...")
                result = model(audio_path, return_timestamps=False)
                print(f"   Model inference complete ({time.time() - start_time:.1f}s)")
            else:
                raise e

        elapsed = time.time() - start_time
        print(f"   ⏱️ Model inference: {elapsed:.1f}s")
        print(f"   📝 Processing & saving segments...")

        # Ensure transcribe folder exists FIRST
        os.makedirs(TRANSCRIBE_FOLDER, exist_ok=True)
        output_path = os.path.join(TRANSCRIBE_FOLDER, f"{video_id}.json")

        save_start = time.time()

        # Process segments with real-time save
        segments = []
        chunk_count = 0

        if "chunks" in result and result["chunks"]:
            # With timestamps - process and save incrementally
            total_chunks = len(result["chunks"])
            print(f"   Processing {total_chunks} chunks (saving incrementally)...")

            for i, chunk in enumerate(result.get("chunks", []), 1):
                if chunk["text"].strip():
                    # Handle missing end timestamp
                    timestamp = chunk.get("timestamp", [None, None])
                    if isinstance(timestamp, tuple):
                        start_ts, end_ts = timestamp
                    else:
                        start_ts = timestamp[0] if len(timestamp) > 0 else None
                        end_ts = timestamp[1] if len(timestamp) > 1 else None

                    # If end timestamp is None, estimate it
                    if end_ts is None and start_ts is not None:
                        # Estimate duration based on text length (rough estimate)
                        estimated_duration = (
                            len(chunk["text"].split()) * 0.4
                        )  # ~0.4s per word
                        end_ts = start_ts + estimated_duration

                    segments.append(
                        {
                            "start": start_ts,
                            "end": end_ts,
                            "text": chunk["text"].strip(),
                        }
                    )
                    chunk_count += 1

                    # Save every 50 chunks for progress feedback
                    if chunk_count % 50 == 0 or i == total_chunks:
                        try:
                            with open(output_path, "w", encoding="utf-8") as f:
                                json.dump(
                                    {"segments": segments},
                                    f,
                                    indent=2,
                                    ensure_ascii=False,
                                )
                            print(
                                f"   💾 Saved {chunk_count}/{total_chunks} segments...",
                                flush=True,
                            )
                        except Exception as e:
                            print(f"⚠️ Write error: {e}", flush=True)

        elif "text" in result:
            # Without timestamps - create single segment
            segments.append({"start": 0.0, "end": None, "text": result["text"].strip()})
            with open(output_path, "w", encoding="utf-8") as f:
                json.dump({"segments": segments}, f, indent=2, ensure_ascii=False)

        if not segments:
            print(f"⚠️ No transcription generated")
            return None

        save_elapsed = time.time() - save_start
        print(f"   ✅ Saved {len(segments)} segments in {save_elapsed:.2f}s")

        # Verify file was created
        if not os.path.exists(output_path):
            print(f"❌ JSON file not found after write!")
            return None

        file_size = os.path.getsize(output_path)
        total_time = time.time() - start_time
        print(f"✅ Transcription complete!")
        print(f"   📊 {len(segments)} segments, {file_size:,} bytes")
        print(
            f"   ⏱️  Total: {total_time:.1f}s (Inference: {elapsed:.1f}s, Save: {save_elapsed:.2f}s)"
        )
        print(f"   💾 {output_path}")
        return output_path
    except torch.cuda.OutOfMemoryError:
        print(
            f"❌ GPU Out of Memory! Try using whisper-base or whisper-tiny", flush=True
        )
        return None
    except Exception as e:
        print(f"❌ Transcription failed: {e}", flush=True)
        import traceback

        traceback.print_exc()
        return None
    finally:
        # Force flush all output
        import sys

        sys.stdout.flush()
        sys.stderr.flush()

============================================================================
Main Workflow Functions
============================================================================

In [ ]:
def download_playlist_to_db(playlist_url: str) -> Tuple[int, int]:
    """
    STEP 1: Download all audio files from playlist and save metadata to database.
    Does NOT transcribe - only downloads audio and saves to DB.

    Args:
        playlist_url: URL of the YouTube playlist

    Returns:
        Tuple of (success_count, failed_count)
    """
    print("=" * 70)
    print("STEP 1: DOWNLOADING ALL AUDIO FILES TO DATABASE")
    print("=" * 70)

    video_links = get_playlist_videos(playlist_url)
    if not video_links:
        print("❌ No videos found in playlist")
        return 0, 0

    print(f"📋 Found {len(video_links)} videos in playlist")
    print(f"📥 Starting download (audio only, no transcription)...")

    conn = connect_db()
    cursor = conn.cursor()

    success, failed, skipped = 0, 0, 0

    pbar = tqdm(
        video_links,
        desc="Downloading",
        unit="video",
        bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]",
    )

    for i, link in enumerate(pbar, 1):
        pbar.set_postfix_str(f"Video {i}")

        # Check if already in database
        if is_video_in_db(cursor, link):
            pbar.set_postfix_str(f"⏭️ {i} in DB")
            skipped += 1
            continue

        # Download audio
        video_id, audio_path, metadata = download_audio(link, TEMP_FOLDER)

        if video_id and audio_path:
            # Save to database (without text_loc)
            try:
                save_audio_metadata(
                    cursor,
                    conn,
                    video_id=video_id,
                    title=metadata["video_name"],
                    youtuber=metadata["youtuber"],
                    date_created=metadata["time_created"],
                    audio_loc=link,  # Use link since audio is stored on web
                    link=link,
                )
                success += 1
                pbar.set_postfix_str(f"✅ {i} saved")
            except Exception as e:
                print(f"\n⚠️ DB error for {video_id}: {e}")
                failed += 1
        else:
            failed += 1
            pbar.set_postfix_str(f"❌ {i} failed")

    pbar.close()
    conn.close()

    print(f"\n{'='*70}")
    print(f"📊 Download Summary:")
    print(f"   ✅ Downloaded: {success}")
    print(f"   ⏭️ Skipped (already in DB): {skipped}")
    print(f"   ❌ Failed: {failed}")
    print(f"{'='*70}\n")

    return success, failed

In [ ]:
def transcribe_all_pending(limit: int = 100) -> Tuple[int, int]:
    """
    STEP 2: Transcribe audio files in database that have audio_loc but no text_loc.
    Only transcribes files that exist in tempvid folder.

    Args:
        limit: Maximum number of videos to transcribe per run (default: 100)

    Returns:
        Tuple of (success_count, failed_count)
    """
    print("=" * 70)
    print("STEP 2: TRANSCRIBING AUDIO FILES WITHOUT TRANSCRIPTION")
    print(f"📊 Limit: {limit} videos per run")
    print("=" * 70)

    # Get videos that need transcription
    pending_videos = get_videos_without_transcription()

    if not pending_videos:
        print("✅ No videos need transcription (all done!)")
        return 0, 0

    # Filter only videos where audio file actually exists
    videos_to_process = []
    for video_id, audio_loc, title, youtuber in pending_videos:
        if audio_loc and os.path.exists(audio_loc):
            videos_to_process.append((video_id, audio_loc, title, youtuber))
        else:
            print(f"⚠️ Audio file not found, skipping: {video_id}")

    if not videos_to_process:
        print("❌ No audio files found in tempvid folder")
        return 0, 0

    # Apply limit
    total_pending = len(videos_to_process)
    videos_to_process = videos_to_process[:limit]

    print(f"📂 Found {total_pending} audio files pending transcription")
    print(f"🎯 Processing {len(videos_to_process)} videos this run (limit: {limit})")
    if total_pending > limit:
        print(f"⏳ Remaining after this run: {total_pending - limit} videos")
    print(f"🎙️ Starting batch transcription...")

    conn = connect_db()
    cursor = conn.cursor()

    success, failed = 0, 0

    pbar = tqdm(
        videos_to_process,
        desc="Transcribing",
        unit="file",
        bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]",
    )

    for video_id, audio_loc, title, youtuber in pbar:
        pbar.set_postfix_str(f"{title[:30]}...")

        # Transcribe
        transcript_path = transcribe(video_id, audio_loc)

        if transcript_path and os.path.exists(transcript_path):
            # Update database with transcription location (relative path)
            rel_transcript_path = os.path.join("data", "transcribe_result", f"{video_id}.json").replace("\\", "/")
            try:
                update_transcription(cursor, conn, video_id, rel_transcript_path)
                success += 1
                pbar.set_postfix_str(f"✅ done")
            except Exception as e:
                print(f"\n⚠️ DB update error for {video_id}: {e}")
                failed += 1
        else:
            failed += 1
            pbar.set_postfix_str(f"❌ failed")

    pbar.close()
    conn.close()

    remaining = total_pending - success - failed

    print(f"\n{'='*70}")
    print(f"📊 Transcription Summary:")
    print(f"   ✅ Transcribed: {success}")
    print(f"   ❌ Failed: {failed}")
    print(f"   ⏳ Remaining: {remaining}")
    if remaining > 0:
        print(f"   💡 Run again to continue transcribing")
    print(f"{'='*70}\n")

    return success, failed

============================================================================
Main Entry Point
============================================================================

In [ ]:
def run_download_in_background(playlist_url: str):
    """Run download in a separate thread."""
    print("\n🔄 Starting download in background thread...")
    download_playlist_to_db(playlist_url)
    print("\n✅ Background download completed!")

In [ ]:
def run_parallel(playlist_url: str, transcribe_limit: int = 100):
    """
    Run download and transcription in parallel.
    - Download runs in background thread
    - Transcription runs in main thread (uses GPU)

    Args:
        playlist_url: YouTube playlist URL to download
        transcribe_limit: Maximum videos to transcribe per run
    """
    print("=" * 70)
    print("🚀 PARALLEL MODE: Download + Transcribe")
    print("=" * 70)
    print("📥 Download will run in background")
    print("🎙️ Transcription will run in main thread (GPU)")
    print("=" * 70)

    # Start download in background thread
    download_thread = threading.Thread(
        target=run_download_in_background, args=(playlist_url,), daemon=True
    )
    download_thread.start()

    # Small delay to let download start
    time.sleep(2)

    # Run transcription in main thread
    print("\n🎙️ Starting transcription in main thread...")
    transcribe_all_pending(limit=transcribe_limit)

    # Wait for download to finish if still running
    if download_thread.is_alive():
        print("\n⏳ Waiting for background download to complete...")
        download_thread.join()

    print("\n" + "=" * 70)
    print("✅ PARALLEL MODE COMPLETED")
    print("=" * 70)

In [ ]:
if __name__ == "__main__":
    # ========================================================================
    # WORKFLOW:
    # ========================================================================
    # 1. download_playlist_to_db(playlist_url)
    #    - Download all audio files from playlist
    #    - Save metadata to database (without transcription)
    #    - No GPU needed, can be done anytime
    #
    # 2. transcribe_all_pending()
    #    - Transcribe all videos in DB that have audio but no transcription
    #    - Only processes files that exist in tempvid folder
    #    - Needs GPU, can be done later when GPU is free
    #
    # Benefits:
    # - Download when internet is good (no GPU needed)
    # - Transcribe later when GPU is free
    # - Easy to resume - just run transcribe_all_pending() again
    # ========================================================================

    # Initialize database
    init_db()

    # Playlist URL
    # PLAYLIST_URL = "https://www.youtube.com/playlist?list=UU0SAXUuHJznVpu3hntun2vA"
    # PLAYLIST_URL_2 = "https://www.youtube.com/playlist?list=UUQo_L_h_01NjLYiNHUnjEJw"

    # ========================================================================
    # Choose ONE of the following modes:
    # ========================================================================

    # === MODE 1: DOWNLOAD ONLY ===
    # download_playlist_to_db(PLAYLIST_URL)
    # download_playlist_to_db(PLAYLIST_URL_2)

    # === MODE 2: TRANSCRIBE ONLY (with limit) ===
    # transcribe_all_pending(limit=TRANSCRIBE_LIMIT)

    # === MODE 3: PARALLEL - Download + Transcribe at the same time ===
    # Download runs in background (uses internet)
    # Transcribe runs in main thread (uses GPU)
    # run_parallel(PLAYLIST_URL, transcribe_limit=TRANSCRIBE_LIMIT)